# 04 — Map of delay count and rate per station

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook provides map visualization of WMATA Rail System sized by delay count and color-coded by delay rate per station

**Sections:**
1. Load features.csv file
2. Calculate delay rate per station
3. Map WMATA Rail System network sized by delay count and color-coded by delay rate per station

In [ ]:
import pandas as pd
import numpy as np

import folium
import folium.plugins as plugins
from folium import CircleMarker

## 1. Load features.csv file

In [ ]:
df = pd.read_csv("../data/features.csv")
df_features = df[df['is_orphan'] == 0]
df_features.head(5)

## 2. Calculate delay rate per station

In [ ]:
delays_df = df_features.groupby(['location_code', 'location_name', 'lat', 'lon']).agg(arrivals=('is_delayed', 'size'), delays=('is_delayed', 'sum')).reset_index()
delays_df['delay_rate'] = (delays_df['delays'] / delays_df['arrivals']).round(2)
delays_df

## 3. Map WMATA Rail System network sized by delay count and color-coded by delay rate per station

In [ ]:
# Create WMATA Rail System map centered on Metro Center station coordinates
lat, lon = 	38.898303, -77.028099
m = folium.Map(
      location=[lat, lon],
      zoom_start=11,
      tiles="OpenStreetMap")

# Define color based on delay rate
def get_color(delay_rate):
    if delay_rate >= 0.30:
        return 'red'
    elif delay_rate >= 0.25:
        return 'orange'
    else:
        return 'green'

# CircleMarkers sized by delay count and color-coded by delay rate per station
for _, row in delays_df.iterrows():
    popup_html = f"""
    <b>{row['location_name']}</b><br>
    Delays: {row['delays']:,}</b><br>
    Delay rate: {row['delay_rate']*100:.2f} %
    """
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['delays'] / 4000,
        popup=folium.Popup(popup_html, max_width=200),
        tooltip=f"Click for {row['location_name']} station details",
        color=get_color(row['delay_rate']),
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

m

In [ ]:
# Save Folium map as a standalone HTML file
m.save('delays_per_station_map.html')